In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw4.ipynb")

---

<h1><center>SDSE Homework 4<br><br> Time series forecasting </center></h1>

---

In this homework you will apply time series forecasting methods to the problem of predicting demands for a large power grid. The data comes from [PJM](https://www.pjm.com/about-pjm), a regional transmission organization covering a large portion of the east coast of the United States. Specifically, the data is for northern Illinois. It contains hourly values of electricity demand (in MW) between 2004 and 2010. 

The homeworks consists of these parts:

+ Load the data
+ Split the data into training, validation, and testing datasets
+ Normalize the data
+ Organize the data into input sequences and output values
+ Train these models
    + Linear regression
    + Multi-layer perceptron (a.k.a. dense neural network)
    + simple RNN
    + LSTM
+ Compare the models
+ Compute performance of the best model

In [ ]:
# NOTE: If you are running this in JupyterLab, you will have to do the following:
# + Uncomment the "pip install" lines below and then run this cell. 
#   This will install CPU-only PyTorch in your JupyterLab environment.
# + Wait until the installation completes.
# + Comment out the lines again, to prevent unnecessary re-installs.
# + Restart your kernel (Kernel>Restart kernel and clear output of all cells...)

# %pip install --upgrade pip
# %pip install --extra-index-url https://download.pytorch.org/whl/cpu torch

In [ ]:
# Do not modify this cell
import os
import random
import numpy as np
import pandas as pd
from pandas.core.indexes.datetimes import DatetimeIndex
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from resources.hashutils import *
os.environ['PYTHONHASHSEED'] = '0'  # optional, for hash-based functions
random.seed(2434)
np.random.seed(2434)
torch.manual_seed(2434)
torch.use_deterministic_algorithms(True)
np.set_printoptions(precision=4)

def predict_model(model, x):
    model.eval()
    x_tensor = torch.tensor(np.asarray(x), dtype=torch.float32)
    with torch.no_grad():
        yhat = model(x_tensor).detach().cpu().numpy()
    return yhat

def predict_any(model, x):
    if isinstance(model, nn.Module):
        return predict_model(model, x)
    return model.predict(x)

# 0. Load the data into a pandas DataFrame

The data is contained in a csv file called `demand.csv` is the `resources` folder. Load this file into a pandas DataFrame using `read_csv` with the following input arguments:
+ index_col=[0]. The first column in the csv file contains the time stamp. This tells pandas to use this as the index of the data frame.
+ parse_dates=[0]. This tells pandas to convert the time stamps into DateTime objects.

Keep the DataFrame in a variable called `raw_data`.

**Note**: This part has already been completed.

In [ ]:
raw_data = pd.read_csv('resources/demand.csv', index_col=[0], parse_dates=[0])

# Display the first 5 rows of the dataset
raw_data.head()

## Plot the full time series

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(raw_data)

# 1. Write data splitting function

Write a function called `split_data` that takes these arguments:
+ `pt`: A number between 0 and 1 corresponding to the proportion of the data to be used for *training*.
+ `pv`: A number between 0 and 1 corresponding to the proportion of the data to be used for *validation*. Validation here means that it will be used by the fitting function of the neural network to evaluate the model at each step of stochastic gradient descent. 

The function should return 3 pandas DataFrames: `data_train`, `data_validate`, and `data_test` (in that order). 

Each of these should contain a **copy** of the segment of `raw_data` that will be used for training, validation, and testing respectively. 

The length of `data_train` should be the integer part of `N*pt`, where `N` is the total number of samples in `raw_data`. Similarly for `data_validate`.

**Notes**:
+ `raw_data` is not passed to `split_data` because it is a global variable.
+ Training data should precede validation data in time. 
+ Validation data should precede testing data in time.

**Hint**:
+ [DataFrame.copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)


In [ ]:
def split_data(pt,pv):

    assert(pt>=0)
    assert(pv>=0)
    assert(pt+pv<=1)

    ...
    
    return data_train, data_validate, data_test

In [ ]:
# test your code:
split_data(pt=0.8,pv=0.1)

In [ ]:
grader.check("q1")

# 2. Split the data

Use your `split_data` function to split the data with the following proportions:
+ `data_train`, with %70 of the data,
+ `data_validate`, with %10 of the data,
+ `data_test`, with %20 of the data.

In [ ]:
data_train, data_validate, data_test = ...

In [ ]:
grader.check("q2")

## Plot the training, validation, and test data. (done already)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(data_train,label='training data')
plt.plot(data_validate,label='validation data')
plt.plot(data_test,label='test data')
plt.legend();

# 3. Normalize the data

Use ScikitLearn's `StandardScaler` to normalize the data. Normalizing a dataset, as we've seen, means subtracting its mean and dividing by its standard deviation. This can improve the performance of certain "scale-dependent" models, such as neural networks. (decision trees are scale-*independent*)

Follow these steps:
1. Create a `StandardScaler` object. You can name it whatever you like (e.g. "scaler").
2. Pass the training data (`data_train`) to its `fit` method. This will compute and store the mean and standard deviation of the training sequence. 
3. Apply the scaling transformation to the training sequence by passing it to the scaler's `transform` method. Save the result in a column called `"scaled"` in the training data DataFrame (`data_train`). 
4. Do the same for `data_validate` and `data_test`.

After doing this, `data_train` should look like this:

<img src="resources/df.png" width=350 />

In [ ]:
from sklearn.preprocessing import StandardScaler

...

In [ ]:
grader.check("q3")

## Plot the scaled training, validation, and test data (done already)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(data_train['scaled'],label='scaled training data')
plt.plot(data_validate['scaled'],label='scaled validation data')
plt.plot(data_test['scaled'],label='scaled test data')
plt.legend();

# 4. Organize the data into input and output sequences

Our goal is to build a model that can forecast the power demand $F$ hours in the future using the $T$ most recent hourly values. To train this model we must organize the training sequence into an input matrix $\mathbf{X}$ and output column vector $\mathbf{Y}$. With $N$ denoting the length of the training sequence (in our case $N=40,915$, as you can verify), the number of rows in $\mathbf{X}$ will be 
$$N' = N-F-T+1$$
The number of columns in $\mathbf{X}$ is $T$.

Write a function called `organize_data` that takes these arguments:
+ `d`: a pandas Series object such as `data_train['scaled']`. 
+ `T`: $T$ as described above.
+ `F`: $F$ as described above.

The function should return the following:
+ `X`, a 2D NumPy array with $N'$ rows and $T$ columns containing the matrix $\mathbf{X}$.
+ `Y`, a 1D NumPy array with $N'$ entries containing the vector $\mathbf{Y}$.
+ `t`, a 1D NumPy array with timestamps for each sample. Each timestamp is a `DatetimeIndex` object.

The two figures below illustrate the organization of $\mathbf{t}$, $\mathbf{X}$, and $\mathbf{Y}$. The first figure shows three input/output samples (green, orange, and blue). The inputs are the sequences of length $T=14$. For example, the green input sequence is collected at 5:00 PM, and it consists of hourly data starting at 4:00 AM. The output of the model is trained to match the forecasted green point at 8:00 PM. 

Similarly, the orange sample consists of an input sequence, collected at 6:00 PM and consisting of  14 hourly values from 5:00 AM to 6:00 PM, and used to predict a value at 9:00 PM.

<img src="resources/samples.png" width=800 />


These samples are arranged into arrays $\mathbf{t}$, $\mathbf{X}$, and $\mathbf{Y}$ as shown below. Notice that the timestamp $\mathbf{t}$ corresponds to the time at which the prediction is made, which is also the time at which the left-most column of $\mathbf{X}$ is measured. For example, the time stamp for the green sample is 5:00 PM.


<img src="resources/snipets.png" width=700 />



In [ ]:
def organize_data(d,F,T):

    N = d.shape[0]
    Np = N - F - T + 1
    X = np.empty((Np,T))
    Y = np.empty(Np)
    t = np.empty(Np,dtype=DatetimeIndex)

    # Fill in X, Y, and t
    ...

    return X, Y, t

In [ ]:
grader.check("q4")

# 5. Call `organize_data` on the training, validation, and test data

We wish to predict the power demand $F=12$ hours in the future from the $T=6$ most recent values. Use `organize_data` to assemble $\mathbf{X}$ and $\mathbf{Y}$ arrays for the **scaled** training, validation, and test data. 

Call these:
+ `Xtrain` and `ytrain` for the training data.
+ `Xvalid` and `yvalid` for the validation data.
+ `Xtest` and `ytest` for the test data.

In [ ]:
T = ...
F = ...
Xtrain, ytrain, ttrain = ...
Xvalid, yvalid, tvalid = ...
Xtest, ytest, ttest = ...

In [ ]:
grader.check("q5")

## Plot the individual sequences (done already)

The plot below shows a few sequences in `Xtrain`, jiggled by a small amount so that they do not overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(9,3))
for i in range(50):
    ax.plot(range(i,i+T),Xtrain[i,:]+np.random.normal(scale=0.02),marker='.',linewidth=1)
ax.spines[['top','right','bottom']].set_visible(False)
ax.grid()

# 6. Linear regression

Train a Scikit-learn linear regression model using `Xtrain` and `ytrain`. 
+ Do not pass any arguments to the `LinearRegression` constructor. 
+ Save the trained model in the variable `linreg`.


In [ ]:
from sklearn.linear_model import LinearRegression

...

In [ ]:
grader.check("q6")

# 7. Performance relative to a baseline model

Let's decide on a performance metric to use. 

Consider the coefficient of determination, $R^2$: 
$$ R^2 = 1 - \frac{\sum(\hat{y}_i-y_i)^2}{\sum(\hat{y}_i-\bar{y})^2}$$
This metric compares the MSE of a model to that of a "baseline" prediction $\bar{y}$, which is simply the mean of the training outputs $\{y_i\}_N$. This makes sense for iid data, but it is too simplistic for time series data. A better baseline for the forecasting problem is to predict that *the future is the same as the present*. In other words, we predict that the value of the power demand in $F$ hours will be the same as the current value. This is illustrated in the figure below for $T=3$ and $F=6$.

<img src="resources/baseline.png" width=700 /> 

The mean absolute error of the model is given by,

$$\text{MAE(model)} = \frac{1}{N'} \sum_{k} \left( |x_{k+F}-\hat{y}_k| \right)    $$

The baseline mean absolute error of the model is,

$$\text{MAE(baseline)} = \frac{1}{N'} \sum_{k} \left( |x_{k+F}-x_k| \right)    $$


The performance of the model is then obtained, analogously to $R^2$ by comparing the MAE of the model to the baseline MAE:

$$ \text{Performance(model)} = 1 - \frac{\text{MAE(model)}}{\text{MAE(baseline)}}$$

Note that the baseline performance is 0, while a "perfect" model (with zero MAE) would have a performance of 1. Better models have larger values of performance.

Write a function called `assess` that implements this performance metric. The function should take these input arguments:
+ `x_current`: a NumPy array with the *current* values of power demand ($x_k$ above). 
+ `y`: a NumPy array with the true future values of power demand ($y_k$ above). 
+ `yhat`: a NumPy array with the model predictions ($\hat{y}_k$ above). 

**Hint**
+ If you implement your own version of MAE, make sure that your y-yhat is a 1D NumPy array and not a large square matrix.
+ To compute the performance of the linear regression model on the validation data we would run

```python 
assess( x_current = Xtrain[:,-1],
        y = ytrain,
        yhat = linreg.predict(Xtrain))
```

In [ ]:
from sklearn.metrics import mean_absolute_error

def assess(x_current,y,yhat):
    ...
    perf = ...
    return perf

In [ ]:
print(f'Performance of linear regression on the training data: {assess(Xtrain[:,-1],ytrain,linreg.predict(Xtrain)):.4f}')
print(f'Performance of linear regression on the validation data: {assess(Xvalid[:,-1],yvalid,linreg.predict(Xvalid)):.4f}')

In [ ]:
grader.check("q7")

# 8. Include hour-of-day

Our linear regression model is performing only about 30\% closer to "perfect prediction" than the baseline model. Let's see if we can improve it.

 We know that demand for electricity is more or less periodic over a day. Hence, hour-of-day may be a useful input to include in the model. To test this idea, we must first write a function that appends the hour-of-day as a final (right-most) column in our input matrix $\mathbf{X}$. 

Write a function called `append_hour_of_day` that takes these input arguments: 
+ `X`: A 2D NumPy array of input sequences, such as `Xtrain`, `Xvalid`, or `Xtest`.
+ `t`: A 1D NumPy array of time stamps, such as `ttrain`, `tvalid`, or `ttest`.

The function should return a new 2D NumPy array with `X` as its first $T$ columns, and the *hour-of-the-day* in its last column.  The hour-of-the-day is an integer between 0 and 23 inclusive. It is stored as the `hour` attribute of each element in `t`, e.g. `ttrain[0].hour`.

**Hint**:

+ Running `append_hour_of_day(Xtrain,ttrain)` should return an array whose first 5 rows are:

<img src="resources/xtrainhour.png" width=600 /> 



In [ ]:
def append_hour_of_day(X,t):
    ...
    return ...

In [ ]:
# test your code
Xtrain_hour = append_hour_of_day(Xtrain,ttrain)
Xtrain_hour[:5,:]

In [ ]:
grader.check("q8")

# 9. Include hour-of-day in training, validation, and test data

Run your `append_hour_of_day` on the training, validation, and test data. Store the results as 
`Xtrain_hour`, `Xvalid_hour`, and `Xtest_hour` respectively.

In [ ]:
...

In [ ]:
grader.check("q9")

# 10. New linear regression with hour-of-day

Train a new linear regression model using the input matrix that includes hour-of-day. Save the fitted model as `linreg_hour`.

Compute the performance of both linear regression models on the validation data. Save the performance of `linreg` to `linreg_perf`, and of `linreg_hour` to `linreg_hour_perf`. Did the performance improve?

In [ ]:
...

In [ ]:
grader.check("q10")

# 11. Build an MLP in PyTorch

We will now try several neural network models to see whether they can deliver better performance than linear regression. We begin with a multi-layer perceptron (MLP).

Build `model_mlp` as a PyTorch `nn.Sequential` model for the forecast problem with the hour-of-the-day as an additional input. The model should have this sequence of layers:

1. `nn.Linear(D, 64)` followed by `nn.ReLU()`
2. `nn.Linear(64, 32)` followed by `nn.ReLU()`
3. `nn.Linear(32, 16)` followed by `nn.ReLU()`
4. `nn.Linear(16, 1)` for the output layer

Use variable name `model_mlp` for this model.

**Notes**
+ Use `D = Xtrain_hour.shape[1]`.
+ Reset the random seed before building the model so that the results are reproducible.
+ Initialize each `nn.Linear` layer with Xavier uniform weights and zero biases.
+ Do not apply an activation after the final output layer.

In [ ]:
random.seed(2434)  # Do not change this. It is needed to ensure repeatability.
torch.manual_seed(2434)

D = Xtrain_hour.shape[1]

model_mlp = nn.Sequential(
    nn.Linear(..., ...),
    nn.ReLU(),
    nn.Linear(..., ...),
    nn.ReLU(),
    nn.Linear(..., ...),
    nn.ReLU(),
    nn.Linear(..., ...)
)

for layer in model_mlp:
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

In [ ]:
grader.check("q11")

# 12. Train the MLP in PyTorch

Train your MLP for 20 epochs using native PyTorch code.

In this part you should:
+ Recreate `model_mlp` from part 11 so that the training starts from the same initialization every time.
+ Convert `Xtrain_hour`, `ytrain`, `Xvalid_hour`, and `yvalid` to `torch.float32` tensors.
+ Create a `TensorDataset` and `DataLoader` for the training data using `batch_size=128` and `shuffle=True`.
+ Define `loss_fn_mlp = nn.MSELoss()`.
+ Define `optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)`.
+ Write a training loop that runs for `20` epochs.
+ After each epoch, compute the training and validation MAE with `mean_absolute_error` and store them in a history dictionary.
+ It will take about 1 minute to train.

Save the history to the variable name `history_mlp`, where
```python
history_mlp = {"epoch": [...], "mae": [...], "val_mae": [...]} 
```

In [ ]:
Xtrain_hour.shape[1]

In [ ]:
random.seed(2434)  # Do not change this. It is needed to ensure repeatability.
torch.manual_seed(2434)

### Copy your code from the previous part here ####

###################################################

for layer in model_mlp:
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

from torch.utils.data import TensorDataset, DataLoader

Xtrain_hour_t = torch.tensor(..., dtype=torch.float32)
ytrain_t = torch.tensor(..., dtype=torch.float32).reshape(-1,1)
Xvalid_hour_t = torch.tensor(..., dtype=torch.float32)
yvalid_t = torch.tensor(..., dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(..., ...)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

loss_fn_mlp = nn.MSELoss()
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)
history_mlp = {"epoch": [], "mae": [], "val_mae": []}

for epoch in range(20):
    model_mlp.train()
    for xb, yb in train_loader:
        optimizer_mlp.zero_grad()
        yhat_train = model_mlp(xb)
        loss = loss_fn_mlp(yhat_train, yb)
        loss.backward()
        optimizer_mlp.step()

    model_mlp.eval()
    with torch.no_grad():
        train_pred = model_mlp(Xtrain_hour_t).cpu().numpy().reshape(-1)
        val_pred = model_mlp(Xvalid_hour_t).cpu().numpy().reshape(-1)

    history_mlp["epoch"].append(epoch)
    history_mlp["mae"].append(mean_absolute_error(ytrain, train_pred))
    history_mlp["val_mae"].append(mean_absolute_error(yvalid, val_pred))

print('Validation: ', assess(Xvalid[:,-1], yvalid, predict_model(model_mlp, Xvalid_hour)))

In [ ]:
assess(Xvalid[:,-1], yvalid, predict_model(model_mlp, Xvalid_hour))

In [ ]:
xv = Xvalid[:,-1]
yhat = predict_model(model_mlp, Xvalid_hour)

In [ ]:
MAE_model = np.mean(np.abs(yvalid - yhat.reshape(-1)))
yvalid.shape, yhat.shape

In [ ]:
grader.check("q12")

## Plot the training history for MLP (done already)

In [ ]:
num_epochs = len(history_mlp['epoch'])
plt.figure(figsize=(9,3))
plt.plot(range(num_epochs), history_mlp['mae'], marker='o', label='Training MAE')
plt.plot(range(num_epochs), history_mlp['val_mae'], marker='o', label='Validation MAE')
plt.xlabel('epoch')
plt.title('Training of MLP')
plt.legend();

# 13. Build and train a SimpleRNN in PyTorch

Next, create a neural network with simple RNN units.

We provide the PyTorch `nn.Module` class for you below. Fill in the missing layer parameters in its `__init__` method so that the model has this structure:
1. `self.rnn1 = nn.RNN(input_size=1, hidden_size=32, batch_first=True)`
2. `self.rnn2 = nn.RNN(input_size=32, hidden_size=32, batch_first=True)`
3. `self.rnn3 = nn.RNN(input_size=32, hidden_size=16, batch_first=True)`
4. `self.output = nn.Linear(16, 1)`

In `forward`, pass the input through the three recurrent layers, keep only the final time step, and then pass it through `self.output`.

Instantiate the model with variable name `model_srnn`, then train it using the provided PyTorch template for 20 epochs. Store the metrics in `history_srnn` with keys `"epoch"`, `"mae"`, and `"val_mae"`.

Use `Xtrain_hour.reshape(-1, D, 1)` and `Xvalid_hour.reshape(-1, D, 1)` when preparing the recurrent-model inputs.

It will take about 3 minutes to train.

In [ ]:
random.seed(2434)  # Do not change this. It is needed to ensure repeatability.
torch.manual_seed(2434)

class SimpleRNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn1 = nn.RNN(input_size=..., hidden_size=..., batch_first=True)
        self.rnn2 = nn.RNN(input_size=..., hidden_size=..., batch_first=True)
        self.rnn3 = nn.RNN(input_size=..., hidden_size=..., batch_first=True)
        self.output = nn.Linear(..., ...)

    def forward(self, x):
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x, _ = self.rnn3(x)
        x = x[:, -1, :]
        x = self.output(x)
        return x

model_srnn = SimpleRNNModel()

Xtrain_hour_seq = torch.tensor(Xtrain_hour.reshape(-1, Xtrain_hour.shape[1], 1), dtype=torch.float32)
ytrain_seq = torch.tensor(ytrain, dtype=torch.float32).reshape(-1,1)
Xvalid_hour_seq = torch.tensor(Xvalid_hour.reshape(-1, Xvalid_hour.shape[1], 1), dtype=torch.float32)
yvalid_seq = torch.tensor(yvalid, dtype=torch.float32).reshape(-1,1)

from torch.utils.data import TensorDataset, DataLoader
train_dataset_srnn = TensorDataset(Xtrain_hour_seq, ytrain_seq)
train_loader_srnn = DataLoader(train_dataset_srnn, batch_size=128, shuffle=True)

optimizer_srnn = optim.Adam(model_srnn.parameters(), lr=0.001)
loss_fn_srnn = nn.MSELoss()
history_srnn = {"epoch": [], "mae": [], "val_mae": []}

for epoch in range(20):
    model_srnn.train()
    for xb, yb in train_loader_srnn:
        optimizer_srnn.zero_grad()
        pred = model_srnn(xb)
        loss = loss_fn_srnn(pred, yb)
        loss.backward()
        optimizer_srnn.step()

    model_srnn.eval()
    with torch.no_grad():
        train_pred = model_srnn(Xtrain_hour_seq).cpu().numpy().reshape(-1)
        valid_pred = model_srnn(Xvalid_hour_seq).cpu().numpy().reshape(-1)

    history_srnn["epoch"].append(epoch)
    history_srnn["mae"].append(mean_absolute_error(ytrain, train_pred))
    history_srnn["val_mae"].append(mean_absolute_error(yvalid, valid_pred))

print('Validation: ', assess(Xvalid[:,-1], yvalid, predict_model(model_srnn, Xvalid_hour.reshape(-1, Xvalid_hour.shape[1], 1))))

In [ ]:
grader.check("q13")

## Plot the training history of the Simple RNN (done already)

In [ ]:
num_epochs = len(history_srnn['epoch'])
plt.figure(figsize=(9,3))
plt.plot(range(num_epochs), history_srnn['mae'], marker='o', label='Training MAE')
plt.plot(range(num_epochs), history_srnn['val_mae'], marker='o', label='Validation MAE')
plt.xlabel('epoch')
plt.title('Training of Simple RNN')
plt.legend();

# 14. Build and train an LSTM in PyTorch

Finally, create a neural network with LSTM units.

We provide the PyTorch `nn.Module` class for you below. Fill in the missing layer parameters in its `__init__` method so that the model has this structure:
1. `self.lstm1 = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)`
2. `self.lstm2 = nn.LSTM(input_size=16, hidden_size=16, batch_first=True)`
3. `self.lstm3 = nn.LSTM(input_size=16, hidden_size=8, batch_first=True)`
4. `self.output = nn.Linear(8, 1)`

In `forward`, pass the input through the three recurrent layers, keep only the final time step, and then pass it through `self.output`.

Instantiate the model with variable name `model_lstm`, then train it using the provided PyTorch template for 20 epochs. Store the metrics in `history_lstm` with keys `"epoch"`, `"mae"`, and `"val_mae"`.

Use `Xtrain_hour.reshape(-1, D, 1)` and `Xvalid_hour.reshape(-1, D, 1)` when preparing the recurrent-model inputs.

It will take about 5 minutes to train.

In [ ]:
random.seed(2434)  # Do not change this. It is needed to ensure repeatability.
torch.manual_seed(2434)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=..., hidden_size=..., batch_first=True)
        self.lstm2 = nn.LSTM(input_size=..., hidden_size=..., batch_first=True)
        self.lstm3 = nn.LSTM(input_size=..., hidden_size=..., batch_first=True)
        self.output = nn.Linear(..., ...)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, _ = self.lstm3(x)
        x = x[:, -1, :]
        x = self.output(x)
        return x

model_lstm = LSTMModel()

Xtrain_hour_lstm = torch.tensor(Xtrain_hour.reshape(-1, Xtrain_hour.shape[1], 1), dtype=torch.float32)
ytrain_lstm = torch.tensor(ytrain, dtype=torch.float32).reshape(-1,1)
Xvalid_hour_lstm = torch.tensor(Xvalid_hour.reshape(-1, Xvalid_hour.shape[1], 1), dtype=torch.float32)
yvalid_lstm = torch.tensor(yvalid, dtype=torch.float32).reshape(-1,1)

from torch.utils.data import TensorDataset, DataLoader
train_dataset_lstm = TensorDataset(Xtrain_hour_lstm, ytrain_lstm)
train_loader_lstm = DataLoader(train_dataset_lstm, batch_size=128, shuffle=True)

optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=0.001)
loss_fn_lstm = nn.MSELoss()
history_lstm = {"epoch": [], "mae": [], "val_mae": []}

for epoch in range(20):
    model_lstm.train()
    for xb, yb in train_loader_lstm:
        optimizer_lstm.zero_grad()
        pred = model_lstm(xb)
        loss = loss_fn_lstm(pred, yb)
        loss.backward()
        optimizer_lstm.step()

    model_lstm.eval()
    with torch.no_grad():
        train_pred = model_lstm(Xtrain_hour_lstm).cpu().numpy().reshape(-1)
        valid_pred = model_lstm(Xvalid_hour_lstm).cpu().numpy().reshape(-1)

    history_lstm["epoch"].append(epoch)
    history_lstm["mae"].append(mean_absolute_error(ytrain, train_pred))
    history_lstm["val_mae"].append(mean_absolute_error(yvalid, valid_pred))

print('Validation: ', assess(Xvalid[:,-1], yvalid, predict_model(model_lstm, Xvalid_hour.reshape(-1, Xvalid_hour.shape[1], 1))))

In [ ]:
grader.check("q14")

## Plot the training history of the LSTM (done already)

In [ ]:
num_epochs = len(history_lstm['epoch'])
plt.figure(figsize=(9,3))
plt.plot(range(num_epochs), history_lstm['mae'], marker='o', label='Training MAE')
plt.plot(range(num_epochs), history_lstm['val_mae'], marker='o', label='Validation MAE')
plt.xlabel('epoch')
plt.title('Training of LSTM')
plt.legend();

## Single plot with the training histories of our three neural network models (done already)

In [ ]:
num_epochs = len(history_lstm['epoch'])

plt.figure(figsize=(9,3))
plt.plot(range(num_epochs), history_mlp['val_mae'], '.-', markersize=10, label='MLP validation MAE')
plt.plot(range(num_epochs), history_srnn['val_mae'], '.-', markersize=10, label='sRNN validation MAE')
plt.plot(range(num_epochs), history_lstm['val_mae'], '.-', markersize=10, label='LSTM validation MAE')
plt.title('Validation MAE')
plt.legend();

# 15. Pick the best model

Evaluate the performance of the four models (linear regression with hour-of-day, MLP, simple RNN, and LSTM) using the validation data (again). Store the result in these variable names: 
+ `linreg_hour_perf`
+ `mlp_perf`
+ `srnn_perf`
+ `lstm_perf`

Choose the best one and save it as `best_model`. For example, if the linear regression model were the best one, your would write:
```python 
best_model = linreg_hour
```

In [ ]:
...

In [ ]:
grader.check("q15")

# 16. Final model test performance

Now that we have made a final selection of a model, it is time to use the test dataset. 

Evaluate the performance of `best_model` on the test data. Save the result to the variable `best_perf`.

In [ ]:
...

In [ ]:
grader.check("q16")

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Please submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)